# 06: BERT Fine-tuning (Optional)

**Goal:** Fine-tune BERT for semantic bill classification (optional for neural network component).

**Status:** OPTIONAL. Complete 07_results_comparison without this if time is limited.

## Method
- Model: `bert-base-uncased` or `distilbert-base-uncased` (faster)
- Training: 3 epochs, lr=2e-5, batch_size=16
- Input: Bill title + summary (truncated to 512 tokens)
- Evaluation: 5-fold CV equivalent using Trainer API

In [ ]:
import sys
sys.path.insert(0, "../")

import pandas as pd
import numpy as np
import torch
from transformers import (
    BertForSequenceClassification,
    BertTokenizer,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Imports successful!")
print(f"GPU available: {torch.cuda.is_available()}")

## Step 1: Prepare Data

In [ ]:
# Load data
df = pd.read_csv("../data/processed/bills_speeches_preprocessed.csv")

# Use bill title + summary as input (shorter than full speech)
df['text_input'] = df['title'].fillna('') + ' ' + df['summary'].fillna('')

# Split train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    df['text_input'].values,
    df['passed'].values,
    test_size=0.2,
    random_state=SEED,
    stratify=df['passed'].values
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Train pass rate: {y_train.mean():.1%}, Test pass rate: {y_test.mean():.1%}")

## Step 2: Tokenize & Create Datasets

In [ ]:
# Load tokenizer and model
model_name = "distilbert-base-uncased"  # Faster than bert-base
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

print(f"Loaded {model_name}")

In [ ]:
# Tokenize
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

# Create HF datasets
train_dataset = Dataset.from_dict({
    'text': X_train,
    'label': y_train
})

test_dataset = Dataset.from_dict({
    'text': X_test,
    'label': y_test
})

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print(f"Tokenized {len(train_dataset)} training samples")

## Step 3: Fine-tune

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="../models/bert_checkpoint",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    seed=SEED,
    disable_tqdm=False,
)

print("Training arguments set")

In [ ]:
# Define compute metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, predictions)
    auc = roc_auc_score(labels, predictions)
    return {'accuracy': acc, 'auc': auc}

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Trainer initialized. Starting fine-tuning...")

In [ ]:
# Fine-tune (this may take 5-10 min depending on GPU)
trainer.train()
print("✓ Fine-tuning complete")

## Step 4: Evaluate

In [ ]:
# Evaluate on test set
test_results = trainer.evaluate(test_dataset)
print("\n=== BERT Test Results ===")
for key, value in test_results.items():
    print(f"{key}: {value:.4f}")

# Save
results_df = pd.DataFrame([test_results])
results_df.to_csv("../results/tables/bert_scores.csv", index=False)

**Note:** This notebook is OPTIONAL. The 3 linear/tree models (logistic, LASSO/Ridge, RF) are sufficient for a strong final report. Skip to `07_results_comparison.ipynb` if time is limited.